#### 测试 NLI 任务

1.1 pipeline 测试 bart-large-mnli 

In [ ]:
from transformers import pipeline

# 1. 加载零样本分类器，底层默认用 bart-large-mnli
classifier = pipeline(
    "zero-shot-classification",
    model=model_dir,
    device=0  # 如果用 GPU:0，否则去掉这个参数
)

text   = "How many people would stop supporting Biden if his lawyer told the country that Biden's former personal assistant needed to be brought out at dawn and killed for saying something that contradicted Biden?  ME! Conservative Christians, why are you still behind Trump and his legal team? His lawyer is literally calling for the death of a man who disagrees with Trump. How does that fit into your moral code? Why aren't you speaking out? Why are you ok with allowing this to continue? You're cheering this on. Absolutely disgusting! THIS is why 80 million people said it's time for change. Too many of you are ok with this type of leadership and behavior even though almost all of you claim to be Christians. #TrumpianChristians #FakeChristians #FakeGod #FakeLove #FakeCompassion #FakeBible #FakeCommandments"
labels = ["I support Trump", "I do not support Trump"]

result = classifier(text, candidate_labels=labels, hypothesis_template="This example is about {}.")
print(result)


1.2 非pipeline 测试 bart-large-mnli

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# 设置模型文件路径
# model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'
model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/bert-mini-finetuned-mnli'

# 加载模型和 tokenizer（从本地加载）
nli_model = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# 确保模型运行在 GPU 上（如果可用）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(device)
nli_model.eval()

def check_nli(topic, post):
    """
    使用文本推断模型判断 post 是否符合 topic
    """
    # 构造前提和假设
    premise = post  # 文章内容
    hypothesis = f"This text is about {topic}."  # 主题假设

    # 编码前提和假设（构造 [SEP] 分隔符）
    encoding = tokenizer.encode(premise, hypothesis, return_tensors='pt', truncation_strategy='only_first')

    # 将输入数据转移到正确的设备（GPU 或 CPU）
    encoding = encoding.to(device)

    # 使用模型进行推理
    with torch.no_grad():
        logits = nli_model(encoding)[0]

    # 选择 "entailment" 和 "contradiction" 的logits
    entail_contradiction_logits = logits[:, [0, 2]]

    # 计算 softmax 概率
    probs = entail_contradiction_logits.softmax(dim=1)

    # 获取 "entailment" 类别的概率（标签为真时的概率）
    prob_label_is_true = probs[:, 1].item()

    # 输出推理结果
    return {
        "label": "entailment" if prob_label_is_true > 0.3 else "not entailment",
        "probability": prob_label_is_true
    }

# 示例输入
# topic = "Trump will make america great again"
# topic = "Trump will make america great again"
topic = "I support Trump"
# post = "It's official. Wife and I have voted. In person."
post = " How many people would stop supporting Biden if his lawyer told the country that Biden's former personal assistant needed to be brought out at dawn and killed for saying something that contradicted Biden?  ME! Conservative Christians, why are you still behind Trump and his legal team? His lawyer is literally calling for the death of a man who disagrees with Trump. How does that fit into your moral code? Why aren't you speaking out? Why are you ok with allowing this to continue? You're cheering this on. Absolutely disgusting! THIS is why 80 million people said it's time for change. Too many of you are ok with this type of leadership and behavior even though almost all of you claim to be Christians. #TrumpianChristians #FakeChristians #FakeGod #FakeLove #FakeCompassion #FakeBible #FakeCommandments"

# 调用推理函数
result = check_nli(topic, post)

# 输出推理结果
print(f"推断结果: {result}")

# 判断是否符合
if result["label"] == "entailment" and result["probability"] > 0.3:
    print("post 与 topic 相符合")
else:
    print("post 与 topic 不相符合")


/home/wangshuo/software/anaconda3/envs/iogs/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2869: FutureWarning: The `truncation_strategy` argument is deprecated and will be removed in a future version, use `truncation=True` to truncate examples to a max length. You can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to truncate to the maximal input size of the model (e.g. 512 for Bert).  If you have pairs of inputs, you can give a specific truncation strategy selected among `truncation='only_first'` (will only truncate the first sentence in the pairs) `truncation='only_second'` (will only truncate the second sentence in the pairs) or `truncation='longest_first'` (will iteratively remove tokens from the longest sentence in the pairs).
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


推断结果: {'label': 'entailment', 'probability': 0.4145878851413727}
post 与 topic 相符合


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# 设置模型文件路径
model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/deberta-v2-xxlarge-mnli'

# 加载模型和 tokenizer（从本地加载）
nli_model = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# 确保模型运行在 GPU 上（如果可用）
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
nli_model.to(device)
nli_model.half()  # 使用半精度以节省内存
nli_model.eval()

def check_nli(topic, post):
    """
    使用文本推断模型判断 post 是否符合 topic
    """
    # 构造前提和假设
    premise = post  # 文章内容
    hypothesis = f"This text is about {topic}."  # 主题假设

    # 编码前提和假设（构造 [SEP] 分隔符）
    encoding = tokenizer.encode(premise, hypothesis, return_tensors='pt', truncation_strategy='only_first')

    # 将输入数据转移到正确的设备（GPU 或 CPU）
    encoding = encoding.to(device)

    # 使用模型进行推理
    with torch.no_grad():
        logits = nli_model(encoding)[0]

    # 选择 "entailment" 和 "contradiction" 的logits
    entail_contradiction_logits = logits[:, [0, 2]]

    # 计算 softmax 概率
    probs = entail_contradiction_logits.softmax(dim=1)

    # 获取 "entailment" 类别的概率（标签为真时的概率）
    prob_label_is_true = probs[:, 1].item()

    # 输出推理结果
    return {
        "label": "entailment" if prob_label_is_true > 0.3 else "not entailment",
        "probability": prob_label_is_true
    }

# 示例输入
# topic = "Trump will make america great again"
# topic = "Trump will make america great again"
topic = "I support Trump"
# post = "It's official. Wife and I have voted. In person."
post = " How many people would stop supporting Biden if his lawyer told the country that Biden's former personal assistant needed to be brought out at dawn and killed for saying something that contradicted Biden?  ME! Conservative Christians, why are you still behind Trump and his legal team? His lawyer is literally calling for the death of a man who disagrees with Trump. How does that fit into your moral code? Why aren't you speaking out? Why are you ok with allowing this to continue? You're cheering this on. Absolutely disgusting! THIS is why 80 million people said it's time for change. Too many of you are ok with this type of leadership and behavior even though almost all of you claim to be Christians. #TrumpianChristians #FakeChristians #FakeGod #FakeLove #FakeCompassion #FakeBible #FakeCommandments"

# 调用推理函数
result = check_nli(topic, post)

# 输出推理结果
print(f"推断结果: {result}")

# 判断是否符合
if result["label"] == "entailment" and result["probability"] > 0.3:
    print("post 与 topic 相符合")
else:
    print("post 与 topic 不相符合")


#### 测试NLI模型的准确率

直接给出entailment的概率

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
import torch

# 1. 指向你微调后保存的模型
# model_dir = '/home/wangshuo/resource/AIModels/Finetune/bert-mini-softlabel-nli'
model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/deberta-v2-xxlarge-mnli'
# model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/bart-large-mnli'
# model_dir = '/home/wangshuo/resource/AIModels/Finetune/distil-proxy/electra_entailment_final'

# 1. 先加载配置，明确 num_labels=2
config = AutoConfig.from_pretrained(model_dir, num_labels=3)

# 2. 加载模型和 tokenizer
nli_model = AutoModelForSequenceClassification.from_pretrained(model_dir,config=config,
    ignore_mismatched_sizes=True)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# 3. 准备设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(device)
nli_model.eval()

def check_nli(topic, post, threshold=0.3):
    """
    使用微调后的二分类 NLI 模型判断 post 是否符合 topic
    输出: { label: 'entailment'|'not_entailment', probability: float }
    """
    # 构造前提和假设
    premise = post
    hypothesis = f"This text is about {topic}."

    # 编码
    encoding = tokenizer(premise, hypothesis,
                         return_tensors='pt',
                         truncation='only_first',
                         padding=True,
                         max_length=128)
    encoding = {k: v.to(device) for k, v in encoding.items()}

    # 推理
    with torch.no_grad():
        logits = nli_model(**encoding).logits  # shape [1,2]
        probs = torch.softmax(logits, dim=-1)[0]    # [p_contra, p_entail]

    p_entail = probs[1].item()
    return {
        "label": "entailment" if p_entail > threshold else "not_entailment",
        "probability": p_entail
    }

# —— 示例调用 ——
topic = "I support Trump"
post = "I hate Trump"
# post = " How many people would stop supporting Biden if his lawyer told the country that Biden's former personal assistant needed to be brought out at dawn and killed for saying something that contradicted Biden?  ME! Conservative Christians, why are you still behind Trump and his legal team? His lawyer is literally calling for the death of a man who disagrees with Trump. How does that fit into your moral code? Why aren't you speaking out? Why are you ok with allowing this to continue? You're cheering this on. Absolutely disgusting! THIS is why 80 million people said it's time for change. Too many of you are ok with this type of leadership and behavior even though almost all of you claim to be Christians. #TrumpianChristians #FakeChristians #FakeGod #FakeLove #FakeCompassion #FakeBible #FakeCommandments"
# post='I support Trump,I love Trump'
# post = "Apparently Trump is way down in the polls with women. I truly don't understand that. Are there that many women who want abortions, are there that many mothers who want their children to fight wars, are there that many women who don't mind criminals and drugs coming thru the border, are there that many women willing to have their Social Security and Medicare dwindle because of aid to illegals, are there that many women who don't care that their children are being indoctrinated, are there that many women who want our country to be another Cuba or Venezuela? REALLY? WTF ladies...wake up"

result = check_nli(topic, post)
print(f"推断结果: {result}")
if result["label"] == "entailment":
    print("✅ post 与 topic 相符合")
else:
    print("❌ post 与 topic 不相符合")


推断结果: {'label': 'not_entailment', 'probability': 0.0009193383157253265}
❌ post 与 topic 不相符合


给出各项的概率

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
import torch

# 1. 指向你的 MNLI 模型（保留三分类头）
model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/deberta-v2-xxlarge-mnli'
# model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/deberta-v2-xlarge-mnli'
# model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/bart-large-mnli'
# model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/bert-mini-finetuned-mnli'
# model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/distilbert-base-uncased-finetuned-mnli'
# 2. 加载配置、模型和 tokenizer
#    注意这里指定 num_labels=3，让它按照三分类来加载
config    = AutoConfig.from_pretrained(model_dir, num_labels=3)
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model     = AutoModelForSequenceClassification.from_pretrained(
    model_dir,
    config=config,
    ignore_mismatched_sizes=True  # 如果你之前做过二分类层替换，也能兼容
)
device    = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
def check_nli_all(topic, post):
    """
    返回一个 dict，包含三个类别的概率：
     - contradiction
     - neutral
     - entailment
    """
    premise    = post
    hypothesis = topic

    # 编码成 tensor
    encoding = tokenizer(
        premise,
        hypothesis,
        return_tensors='pt',
        truncation='only_first',
        padding=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        logits = model(**encoding).logits  # shape [1, 3]
        # 原始三分类 softmax
        probs_3 = torch.softmax(logits[0], dim=-1)  # [p_contra, p_neu, p_ent]

        # 只取 contra(0) 和 ent(2) 做二分类归一
        logits_2 = logits[0, [0, 2]]              # [logit_contra, logit_ent]
        probs_2  = torch.softmax(logits_2, dim=-1)  # [p_bc_contra, p_bc_ent]

    return {
        # 三分类
        "contradiction": probs_3[0].item(),
        "neutral":       probs_3[1].item(),
        "entailment":    probs_3[2].item(),
        # 二分类（renormed）
        "bc_contradiction": probs_2[0].item(),
        "bc_entailment":    probs_2[1].item()
    }

In [45]:
# —— 示例调用 ——  
topic = "I support Trump"
# hypothesis = f"This text is about {topic}."
hypothesis = f"This text is about {topic}."
post0  = "I hate Trump"
post1 = " How many people would stop supporting Biden if his lawyer told the country that Biden's former personal assistant needed to be brought out at dawn and killed for saying something that contradicted Biden?  ME! Conservative Christians, why are you s` till behind Trump and his legal team? His lawyer is literally calling for the death of a man who disagrees with Trump. How does that fit into your moral code? Why aren't you speaking out? Why are you ok with allowing this to continue? You're cheering this on. Absolutely disgusting! THIS is why 80 million people said it's time for change. Too many of you are ok with this type of leadership and behavior even though almost all of you claim to be Christians. #TrumpianChristians #FakeChristians #FakeGod #FakeLove #FakeCompassion #FakeBible #FakeCommandments"
post2 = "Apparently Trump is way down in the polls with women. I truly don't understand that. Are there that many women who want abortions, are there that many mothers who want their children to fight wars, are there that many women who don't mind criminals and drugs coming thru the border, are there that many women willing to have their Social Security and Medicare dwindle because of aid to illegals, are there that many women who don't care that their children are being indoctrinated, are there that many women who want our country to be another Cuba or Venezuela? REALLY? WTF ladies...wake up"
# post3= "Holy shit!!! Trump is winning Virginia!!!"
post3= "If you're feeling down about the election, if you're feeling like the clock is running out on President Trump, if you're worried about the States certifying the election count for Biden I urge you to watch this video!  President Trump has this in the bag."
post4="In with Trump all the way"
scores = check_nli_all(hypothesis, post3)
print("三分类概率：", scores)

三分类概率： {'contradiction': 0.07994070649147034, 'neutral': 0.5868690609931946, 'entailment': 0.33319026231765747, 'bc_contradiction': 0.19349966943264008, 'bc_entailment': 0.8065003156661987}


post0:
deberta-v2-xxlarge-mnli：  {'contradiction': 0.998138427734375, 'neutral': 0.0009193383157253265, 'entailment': 0.0009423050796613097}
deberta-v2-xlarge-mnli ： {'contradiction': 0.9985392093658447, 'neutral': 0.0005599606083706021, 'entailment': 0.0009008939960040152}
bart-large-mnli ：三分类概率： {'contradiction': 0.9989224672317505, 'neutral': 0.0005110081983730197, 'entailment': 0.000566595233976841}

post1:
deberta-v2-xxlarge-mnli： {'contradiction': 0.9729936122894287, 'neutral': 0.022142112255096436, 'entailment': 0.004864322952926159}
deberta-v2-xlarge-mnli ： {'contradiction': 0.9985392093658447, 'neutral': 0.0005599606083706021, 'entailment': 0.0009008939960040152}
bart-large-mnli ：三分类概率： {'contradiction': 0.9989224672317505, 'neutral': 0.0005110081983730197, 'entailment': 0.000566595233976841}

post2:
deberta-v2-xxlarge-mnli：{'contradiction': 0.06582507491111755, 'neutral': 0.6019459962844849, 'entailment': 0.3322289288043976}
deberta-v2-xlarge-mnli ： {'contradiction': 0.9985392093658447, 'neutral': 0.0005599606083706021, 'entailment': 0.0009008939960040152}
bart-large-mnli ：三分类概率： {'contradiction': 0.9989224672317505, 'neutral': 0.0005110081983730197, 'entailment': 0.000566595233976841}

测试文本相似度

In [20]:
from sentence_transformers import SentenceTransformer, util

# 1. 选一个常用的 SBERT 模型
# model = SentenceTransformer('/home/wangshuo/resource/AIModels/NLP/TSI/all-MiniLM-L6-v2')
# model = SentenceTransformer('/home/wangshuo/resource/AIModels/NLP/TSI/all-mpnet-base-v2')
# model = SentenceTransformer('/home/wangshuo/resource/AIModels/NLP/TSI/gtr-t5-large')
model = SentenceTransformer('/home/wangshuo/resource/AIModels/NLP/TSI/LaBSE')

# 2. 准备文本
short = "Support Trump"
# long = "TRUMP'S AMERICA"
# long = "trump is rubbish"
long = "I hate Trump"
# short = "Apparently Trump is way down in the polls with women. I truly don't understand that. Are there that many women who want abortions, are there that many mothers who want their children to fight wars, are there that many women who don't mind criminals and drugs coming thru the border, are there that many women willing to have their Social Security and Medicare dwindle because of aid to illegals, are there that many women who don't care that their children are being indoctrinated, are there that many women who want our country to be another Cuba or Venezuela? REALLY? WTF ladies...wake up"
# long = " How many people would stop supporting Biden if his lawyer told the country that Biden's former personal assistant needed to be brought out at dawn and killed for saying something that contradicted Biden?  ME! Conservative Christians, why are you s` till behind Trump and his legal team? His lawyer is literally calling for the death of a man who disagrees with Trump. How does that fit into your moral code? Why aren't you speaking out? Why are you ok with allowing this to continue? You're cheering this on. Absolutely disgusting! THIS is why 80 million people said it's time for change. Too many of you are ok with this type of leadership and behavior even though almost all of you claim to be Christians. #TrumpianChristians #FakeChristians #FakeGod #FakeLove #FakeCompassion #FakeBible #FakeCommandments"

# 3. 编码（自动做 tokenization + pooling）
emb1 = model.encode(short,  convert_to_tensor=True)
emb2 = model.encode(long,   convert_to_tensor=True)

# 4. 计算余弦相似度
cos_sim = util.pytorch_cos_sim(emb1, emb2)
print("Cosine Similarity:", cos_sim.item())


Cosine Similarity: 0.6640014052391052
